In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import joblib



In [3]:
# Load features
features_clean = pd.read_parquet("../data/processed/features.parquet")

# Define feature columns and targets
FEATURE_COLS = [
    "momentum", "return_1m", "volatility_21d", "drawdown_52w",
    "ma_ratio_200", "rsi_14", "volume_ratio_20", "beta_252", "vix"
]

# Train/test split at 2023
train = features_clean[features_clean.index.get_level_values("date") < "2023-01-01"]
test  = features_clean[features_clean.index.get_level_values("date") >= "2023-01-01"]

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

# Prepare arrays for Return RF
X_train = train[FEATURE_COLS]
y_train_ret = train["forward_return"]
y_train_vol = train["forward_volatility"]

X_test = test[FEATURE_COLS]
y_test_ret = test["forward_return"]
y_test_vol = test["forward_volatility"]


Train: (167742, 11)
Test:  (78120, 11)


In [4]:
# Return RF
rf_return = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1      # use all CPU cores
)
rf_return.fit(X_train, y_train_ret)
print("Return RF trained.")

# Volatility RF
rf_volatility = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf_volatility.fit(X_train, y_train_vol)
print("Volatility RF trained.")


Return RF trained.
Volatility RF trained.


In [5]:
# Predictions
pred_ret = rf_return.predict(X_test)
pred_vol = rf_volatility.predict(X_test)

# Return RF metrics
r2_ret  = r2_score(y_test_ret, pred_ret)
mse_ret = mean_squared_error(y_test_ret, pred_ret)
rmse_ret = np.sqrt(mse_ret)

# Volatility RF metrics
r2_vol  = r2_score(y_test_vol, pred_vol)
rmse_vol = np.sqrt(mean_squared_error(y_test_vol, pred_vol))

print(f"Return RF    — R²: {r2_ret:.4f}, RMSE: {rmse_ret:.4f}")
print(f"Volatility RF — R²: {r2_vol:.4f}, RMSE: {rmse_vol:.4f}")


Return RF    — R²: -0.0908, RMSE: 0.0813
Volatility RF — R²: 0.0131, RMSE: 0.1116


In [6]:
from scipy.stats import spearmanr

rho_ret, _ = spearmanr(y_test_ret, pred_ret)
rho_vol, _ = spearmanr(y_test_vol, pred_vol)

print(f"Return RF    — Spearman ρ: {rho_ret:.4f}")
print(f"Volatility RF — Spearman ρ: {rho_vol:.4f}")


Return RF    — Spearman ρ: 0.0062
Volatility RF — Spearman ρ: 0.4618


In [7]:
feat_importance = pd.Series(
    rf_return.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

print(feat_importance)


beta_252           0.143854
vix                0.131472
ma_ratio_200       0.126249
volatility_21d     0.121014
drawdown_52w       0.118387
momentum           0.115609
return_1m          0.090019
rsi_14             0.086091
volume_ratio_20    0.067304
dtype: float64


In [8]:
pred_ret_train = rf_return.predict(X_train)
r2_train = r2_score(y_train_ret, pred_ret_train)
print(f"Return RF — Train R²: {r2_train:.4f}, Test R²: {r2_ret:.4f}")


Return RF — Train R²: 0.9379, Test R²: -0.0908


In [ ]:

param_grid = {
    "max_depth":        [5, 10, 20, None],
    "min_samples_leaf": [10, 50, 100, 200]
}

tscv = TimeSeriesSplit(n_splits=5)

rf_tuned = RandomForestRegressor(
    n_estimators=100,      # faster for search
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

grid_search = GridSearchCV(
    rf_tuned,
    param_grid,
    cv=tscv,
    scoring="r2",
    verbose=1
)

grid_search.fit(X_train, y_train_ret)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV R²: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best parameters: {'max_depth': 5, 'min_samples_leaf': 200}
Best CV R²: -0.0361


In [10]:
rf_return_final = RandomForestRegressor(
    n_estimators=500,
    max_depth=5,
    min_samples_leaf=200,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf_return_final.fit(X_train, y_train_ret)

pred_ret_final = rf_return_final.predict(X_test)
r2_final = r2_score(y_test_ret, pred_ret_final)
rho_final, _ = spearmanr(y_test_ret, pred_ret_final)
r2_train_final = r2_score(y_train_ret, rf_return_final.predict(X_train))

print(f"Train R²: {r2_train_final:.4f}")
print(f"Test R²:  {r2_final:.4f}")
print(f"Spearman ρ: {rho_final:.4f}")


Train R²: 0.0627
Test R²:  0.0023
Spearman ρ: 0.0647
